In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files ar+++++++++++++++++++++++++++++++++e available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score

import lightgbm as lgb

# =====================
# LOAD DATA
# =====================
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv")

print("✅ Data Loaded")

# =====================
# CLEANING
# =====================
train["TotalCharges"] = pd.to_numeric(train["TotalCharges"], errors="coerce")
test["TotalCharges"] = pd.to_numeric(test["TotalCharges"], errors="coerce")

train["TotalCharges"] = train["TotalCharges"].fillna(train["TotalCharges"].median())
test["TotalCharges"] = test["TotalCharges"].fillna(test["TotalCharges"].median())

# Normalize categories
for col in ["OnlineSecurity","OnlineBackup","DeviceProtection",
            "TechSupport","StreamingTV","StreamingMovies"]:
    train[col] = train[col].replace({"No internet service": "No"})
    test[col] = test[col].replace({"No internet service": "No"})

train["MultipleLines"] = train["MultipleLines"].replace({"No phone service": "No"})
test["MultipleLines"] = test["MultipleLines"].replace({"No phone service": "No"})

# =====================
# TARGET
# =====================
y = train["Churn"].map({"Yes":1, "No":0})

# =====================
# FEATURE ENGINEERING (STRONG ONLY)
# =====================
train["avg_charge"] = train["TotalCharges"] / (train["tenure"] + 1)
test["avg_charge"] = test["TotalCharges"] / (test["tenure"] + 1)

train["monthly_per_tenure"] = train["MonthlyCharges"] / (train["tenure"] + 1)
test["monthly_per_tenure"] = test["MonthlyCharges"] / (test["tenure"] + 1)

# =====================
# 🔥 K-FOLD TARGET ENCODING (SAFE)
# =====================
def target_encode(train, test, col, y):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    train_encoded = np.zeros(len(train))
    
    for tr_idx, val_idx in kf.split(train):
        X_tr, X_val = train.iloc[tr_idx], train.iloc[val_idx]
        y_tr = y.iloc[tr_idx]
        
        temp = pd.DataFrame({col: X_tr[col], "target": y_tr})
        mean_map = temp.groupby(col)["target"].mean()
        
        train_encoded[val_idx] = X_val[col].map(mean_map)
    
    global_map = pd.DataFrame({col: train[col], "target": y}).groupby(col)["target"].mean()
    test_encoded = test[col].map(global_map)
    
    return train_encoded, test_encoded

# Apply ONLY to high-impact columns
for col in ["Contract", "PaymentMethod", "InternetService"]:
    train[col+"_te"], test[col+"_te"] = target_encode(train, test, col, y)

# =====================
# DROP UNUSED
# =====================
test_ids = test["id"]

train.drop(["Churn","id","Contract","PaymentMethod","InternetService"], axis=1, inplace=True)
test.drop(["id","Contract","PaymentMethod","InternetService"], axis=1, inplace=True)

# =====================
# REMAINING CATEGORICAL
# =====================
cat_cols = train.select_dtypes(include="object").columns

for col in cat_cols:
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")

print("✅ Preprocessing Done")

# =====================
# MODEL (TUNED)
# =====================
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros(len(train))
preds = np.zeros(len(test))

print("🚀 Training Started...")

for fold, (tr_idx, val_idx) in enumerate(kf.split(train, y)):
    
    print(f"🔥 Fold {fold+1}")
    
    X_tr, X_val = train.iloc[tr_idx], train.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.01,
        num_leaves=96,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.2,
        reg_lambda=0.2,
        min_child_samples=20,
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(100)]
    )
    
    oof[val_idx] = model.predict_proba(X_val)[:,1]
    preds += model.predict_proba(test)[:,1] / 5

print("\n✅ CV AUC:", roc_auc_score(y, oof))

# =====================
# SUBMISSION
# =====================
submission = sample.copy()
submission[sample.columns[1]] = preds.astype(float)

submission.to_csv("submission.csv", index=False)



✅ Data Loaded
✅ Preprocessing Done
🚀 Training Started...
🔥 Fold 1
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.038675 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1184
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1500]	valid_0's auc: 0.915733	valid_0's binary_logloss: 0.2989
🔥 Fold 2
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.044181 seconds.
You can set `